# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
meta = dataset.metadata

print("Dataset Title:", meta.name)
print("Description:", meta.description)
print("License:", meta.license)
print("Published:", meta.datePublished)
print("Conforms to Croissant spec:", meta.conformsTo)

## 2. Data Overview
Review available record sets, their fields and columns, and their `@id`s.

We will print all record sets declared in the dataset and their available fields/columns, referencing them by their `@id`.

In [ ]:
# List all record sets and their fields/columns by their @id
def get_record_sets(meta):
    # Some Croissant schemas may use 'recordSet' (list) or 'recordSets', check both
    if hasattr(meta, 'recordSet') and meta.recordSet:
        return meta.recordSet
    elif hasattr(meta, 'recordSets') and meta.recordSets:
        return meta.recordSets
    else:
        return []

record_sets = get_record_sets(meta)

if not record_sets:
    print("No record sets found directly in metadata. Attempting to infer them from attached distributions...")
    # Fallback: in some datasets, recordSets are not top-level but implied from files or distributions
    # We can use dataset.record_set_ids (added by mlcroissant) if available
    record_set_ids = getattr(dataset, 'record_set_ids', None)
    if record_set_ids and len(record_set_ids) > 0:
        record_sets = record_set_ids
    else:
        # Try discovery via dataset.records() generator
        try:
            all_ids = dataset.record_set_ids
            if all_ids:
                record_sets = all_ids
        except Exception as e:
            print('Could not determine record sets:', e)
            record_sets = []

print(f"Found {len(record_sets)} record set(s).")
for rs in record_sets:
    print(f'  - Record set @id: {rs}')
    # List the available field/column ids for each record set
    try:
        df_preview = pd.DataFrame(dataset.records(record_set=rs))
        print(f"    Columns/fields (@id): {list(df_preview.columns)}")
    except Exception as e:
        print(f"    Could not preview records for record set '{rs}': {e}")

## 3. Data Extraction
Let's extract data from one or more record sets into DataFrames for analysis.
Below, we extract data using the record set and field/column `@id`s identified above.

In [ ]:
# Choose which record set(s) by @id to extract
record_set_ids = record_sets  # You can filter/choose specific ones if you wish
dataframes = {}
for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        if len(records) == 0:
            print(f"No records found for record set {rsid}")
        else:
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"Loaded DataFrame for {rsid} with {len(df)} rows and {len(df.columns)} columns.")
            print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for record set {rsid}: {e}")

# Preview the first few rows for the first record set (if any)
if len(dataframes) > 0:
    sample_rsid = list(dataframes.keys())[0]
    print(f"\nPreview from record set {sample_rsid}:")
    display(dataframes[sample_rsid].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria and normalizing numeric fields. Operations include removing outliers, transforming data distributions, or grouping data by key attributes for further analysis.

_For demonstration, we select a numeric field from the first loaded record set (by @id) and process it._

In [ ]:
import numpy as np

# Select main DataFrame and find available numeric columns
if len(dataframes) == 0:
    raise RuntimeError("No DataFrames loaded; cannot perform EDA.")

main_rsid = list(dataframes.keys())[0]
df = dataframes[main_rsid]

# Identify numeric field(s)
numeric_candidates = df.select_dtypes(include=np.number).columns.tolist()
if len(numeric_candidates) == 0:
    # Try to convert columns named like typical numeric fields (case insensitive matches)
    preferred = [col for col in df.columns if any(x in col.lower() for x in ["abundance", "mw", "peptide", "coverage", "count", "pI", "intensity", "mod", "score"])]
    # Attempt conversion
    for col in preferred:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    numeric_candidates = df.select_dtypes(include=np.number).columns.tolist()

print("Numeric candidate fields (by @id):", numeric_candidates)
if not numeric_candidates:
    raise RuntimeError("No numeric fields available for EDA.")

# For demo, pick the first numeric field by @id
numeric_field_id = numeric_candidates[0]

# Example threshold (set by user or auto)
threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records in '{main_rsid}' with field @{numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize selected numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a potential categorical field, if available
categorical_cols = df.select_dtypes(include='object').columns.tolist()
group_field_id = None
for col in categorical_cols:
    # We try to find a meaningful field like 'description', 'sample', 'accession', etc.
    if any(x in col.lower() for x in ['sample', 'accession', 'description', 'group', 'type', 'condition']):
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"Grouped data by @{group_field_id} (mean of {numeric_field_id}):")
    display(grouped_df.head())
else:
    print('No suitable grouping field found for demo.')

## 5. Visualization
Visualize the distribution of the main numeric field and (if available) the group means.

We show a histogram of the selected numeric field and, if grouped, a bar chart of group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

if group_field_id:
    plt.figure(figsize=(10,5))
    group_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)[:20]
    sns.barplot(y=group_means.index, x=group_means.values)
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.title(f"Top 20 {group_field_id} by mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We have loaded the Mass Spectrometry Analysis of Extracellular Vesicles dataset using the Croissant schema and `mlcroissant`. By referencing all record sets and fields by their `@id`, we demonstrated:
- How to enumerate available record sets and columns
- Loading records into DataFrames for analysis
- Discovering and normalizing numeric fields for basic exploratory analysis
- Grouping and visualizing attributes using field `@id`s

This notebook provides a reproducible workflow for FAIR dataset exploration and can be adapted for additional biological/biomedical analysis tasks.